<a href="https://colab.research.google.com/github/eilselx/MACHINE-LEARNING-PROJECT---LIFE-EXPECTANCY-ESTIMATOR-/blob/main/MACHINE_LEARNING_PROJECT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

The First Ever Machine Learning Project!                 


Problem Statement: Can machine learning accurately predict a country's life expectancy using demographic, healthcare, economic, and public health indicators?

Section 1 - By Lavanya & Sikhin:

Data Cleaning, Preprocessing, Exploratory Data Analysis (EDA), Data Visualization


In [ ]:
from google.colab import files
import zipfile
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import os


In [ ]:
!git clone https://github.com/eilselx/MACHINE-LEARNING-PROJECT---LIFE-EXPECTANCY-ESTIMATOR-.git

In [ ]:
!ls


In [ ]:
%cd MACHINE-LEARNING-PROJECT---LIFE-EXPECTANCY-ESTIMATOR-

In [ ]:
!ls

In [ ]:
%cd data

In [ ]:
!ls

In [ ]:
import zipfile
from pathlib import Path

# Define the zip file and extraction directory
zip_file = Path("archive(1).zip")
extract_dir = Path(".")

# Extract the dataset only if it hasn't been extracted already
if not (extract_dir / "health_indicators.csv").exists():
    with zipfile.ZipFile(zip_file, "r") as zip_ref:
        zip_ref.extractall(extract_dir)
    print("Dataset extracted successfully.")
else:
    print("Dataset already extracted. Skipping extraction.")

In [ ]:
df = pd.read_csv("health_indicators.csv")

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
display(df.describe())


In [ ]:
missing_percent = (df.isnull().sum() / len(df)) * 100

missing_percent.sort_values(ascending=False)

In [ ]:
df_clean= df.copy()

In [ ]:
# Columns to remove due to excessive missing values
columns_to_drop = [
    "obesity_pct",
    "diabetes_pct",
    "prenatal_care_pct",
    "overweight_children_pct",
    "wasting_pct",
    "stunting_pct"
]

# Drop the columns
df_clean.drop(columns=columns_to_drop, inplace=True)

# Check the new shape
df_clean.shape

## Handling missing values: Health Infrastucture

In [ ]:
def hierarchical_imputation(df, columns):
    """
    Hierarchical imputation:
    1. Fill within the same country (forward & backward fill)
    2. Fill remaining values with regional median
    3. Fill remaining values with global median
    """

    for col in columns:

        # Step 1: Country-level fill
        df[col] = (
            df.groupby("country_name")[col]
            .transform(lambda x: x.ffill().bfill())
        )

        # Step 2: Regional median
        df[col] = df[col].fillna(
            df.groupby("region")[col].transform("median")
        )

        # Step 3: Global median
        df[col] = df[col].fillna(
            df[col].median()
        )

    return df

In [ ]:
healthcare_cols = [
    "physicians_per_1000",
    "hospital_beds_per_1000",
    "nurses_per_1000"
]

df_clean = hierarchical_imputation(df_clean, healthcare_cols)

In [ ]:
water_cols = [
    "safe_water_pct",
    "sanitation_pct"
]

df_clean = hierarchical_imputation(df_clean, water_cols)

In [ ]:
disease_cols = [
    "hiv_incidence",
    "maternal_mortality",
    "communicable_death_pct",
    "noncommunicable_death_pct"
]

df_clean = hierarchical_imputation(df_clean, disease_cols)

In [ ]:
lifestyle_cols = [
    "smoking_male",
    "smoking_female",
    "alcohol_per_capita"
]

df_clean = hierarchical_imputation(df_clean, lifestyle_cols)

In [ ]:
economic_cols = [
    "health_expenditure_pct_gdp",
    "health_expenditure_per_capita",
    "gdp_per_capita",
    "gdp_per_capita_ppp",
    "poverty_rate"
]

df_clean = hierarchical_imputation(df_clean, economic_cols)

In [ ]:
df_clean.isnull().sum().sort_values(ascending=False)

In [ ]:
df_clean.drop(columns=["handwashing_pct"], inplace=True)

In [ ]:
remaining_cols = [
    "population_growth",
    "tb_incidence",
    "under5_mortality",
    "neonatal_mortality",
    "immunization_measles",
    "immunization_dpt",
    "immunization_pol3",
    "immunization_bcg",
    "immunization_hib3",
    "undernourishment_pct"
]

df_clean = hierarchical_imputation(df_clean, remaining_cols)

In [ ]:
df_clean.isnull().sum().sort_values(ascending=False)

In [ ]:
# Number of duplicate rows
df_clean.duplicated().sum()

In [ ]:
df_clean.info()

In [ ]:
df_clean.to_csv("health_indicators_cleaned.csv", index=False)

## EDA
